# QLoRA fine-tune: Llama 3 8B for CEFR classification

Trains a QLoRA adapter on `meta-llama/Meta-Llama-3-8B` to classify the CEFR
difficulty (A1 to C2) of an English sentence, selects the best epoch by
validation macro F1, predicts the benchmark test set, pushes the adapter to
the Hugging Face Hub, and exports prediction files for the eval harness in
https://github.com/akallam04/cefr-qlora-benchmark

Before running:
1. Runtime > Change runtime type > select a GPU. T4 (free) works; L4 or A100 are faster.
2. Colab Secrets (key icon in the left sidebar): add `HF_TOKEN` (your current Hugging Face Write token) and `WANDB_API_KEY`, and enable notebook access for both.
3. Your Hugging Face account must have accepted the Meta Llama 3 license.

Expected runtime for 3 epochs: T4 roughly 1.5 to 2.5 h, L4 roughly 40 to 70 min, A100 roughly 20 to 35 min, plus a one-time download of about 16 GB of base weights.

Run cells top to bottom. Training only starts after two safety cells have proven the label tokenization and the loss masking are correct. If the runtime disconnects, reconnect and run all cells again: training resumes from the last epoch checkpoint (on the free tier set `USE_DRIVE = True` in the config cell so checkpoints survive).

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > select a GPU."
GPU_NAME = torch.cuda.get_device_name(0)
BF16_OK = torch.cuda.is_bf16_supported()
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU_NAME} | VRAM: {VRAM_GB:.1f} GB | bf16 supported: {BF16_OK}")

## Install the pinned training stack

Exact pins for reproducibility. Torch is deliberately not reinstalled: Colab
ships a CUDA build matched to its drivers.

In [ ]:
%pip install -q transformers==5.13.0 trl==1.7.1 peft==0.19.1 bitsandbytes==0.49.2 accelerate==1.14.0 datasets==5.0.0 wandb==0.28.0 scikit-learn==1.9.0

## Credentials

Reads both keys from Colab Secrets and fails with instructions if they are
missing. `auth_check` proves the token can reach the gated Llama 3 repo
before any download starts.

In [ ]:
import os

from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    WANDB_KEY = userdata.get("WANDB_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError) as exc:
    raise RuntimeError(
        "Missing Colab secret. Open the key icon in the left sidebar, add "
        "HF_TOKEN and WANDB_API_KEY, and toggle notebook access on for both."
    ) from exc

from huggingface_hub import auth_check, login

login(token=HF_TOKEN)
auth_check("meta-llama/Meta-Llama-3-8B", token=HF_TOKEN)
print("Hugging Face login ok, gated Llama 3 access confirmed")

import wandb

wandb.login(key=WANDB_KEY)
os.environ["WANDB_PROJECT"] = "cefr-qlora-benchmark"

## Get the benchmark repo and rebuild the dataset

`prepare_data.py` re-downloads CEFR-SP pinned to the exact upstream commit
and verifies every byte against the committed checksum manifest, so this
notebook provably trains on the same data the baseline was evaluated on.

In [ ]:
%cd /content
import sys

!git -C cefr-qlora-benchmark pull --quiet 2>/dev/null || git clone --quiet https://github.com/akallam04/cefr-qlora-benchmark.git
%cd /content/cefr-qlora-benchmark
sys.path.insert(0, "src")

!python src/prepare_data.py

## Configuration

Everything tunable lives here. LoRA rank 16 with alpha 32 on all seven
attention and MLP projections is the standard QLoRA recipe; the batch size
adapts to the GPU (bf16 GPUs fit more).

In [ ]:
CONFIG = {
    "model_id": "meta-llama/Meta-Llama-3-8B",
    "adapter_repo": "akallam04/Llama-3-8B-cefr-qlora",
    "max_length": 128,   # longest sentence in the corpus is 34 words, audited below
    "lora_r": 16,        # rank of the adapter matrices
    "lora_alpha": 32,    # adapter scaling: effective multiplier is alpha / r = 2
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "epochs": 3,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "effective_batch": 16,
    "seed": 42,
}

USE_DRIVE = False  # free tier: set True so checkpoints survive disconnects
OUTPUT_DIR = "/content/qlora-out"
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/cefr-qlora-out"

PER_DEVICE = 8 if BF16_OK else 4
GRAD_ACCUM = CONFIG["effective_batch"] // PER_DEVICE
print(f"per-device batch {PER_DEVICE} x grad accum {GRAD_ACCUM} = effective batch {CONFIG['effective_batch']}")
print(f"checkpoints go to {OUTPUT_DIR}")

## Build prompt-completion datasets

The prompt template comes from `src/utils.py`, the same module the
prediction script uses, so training and inference can never drift apart.
TRL computes loss on the completion only for this dataset format, and the
completion is a single digit token.

In [ ]:
import pandas as pd
from datasets import Dataset

from utils import completion_for, format_prompt

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
test_df = pd.read_csv("data/processed/test.csv")


def to_example(row):
    return {"prompt": format_prompt(row["sentence"]), "completion": completion_for(row["label"])}


train_ds = Dataset.from_list([to_example(r) for _, r in train_df.iterrows()]).shuffle(seed=CONFIG["seed"])
val_ds = Dataset.from_list([to_example(r) for _, r in val_df.iterrows()])
print(train_ds[0])
print(f"{len(train_ds)} train and {len(val_ds)} val examples")

## Safety cell 1: tokenizer facts

Asserts each label is exactly one token (the whole design rests on this)
and that the longest prompt fits inside `max_length` with room to spare.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(CONFIG["model_id"], token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

LABEL_IDS = {}
for digit in "123456":
    ids = tok.encode(" " + digit, add_special_tokens=False)
    assert len(ids) == 1, f"label ' {digit}' tokenizes to {ids}: single-token design broken"
    LABEL_IDS[int(digit)] = ids[0]
print("label token ids:", LABEL_IDS)

longest_rows = train_df["sentence"].str.len().nlargest(20).index
longest_prompt = max(
    len(tok.encode(format_prompt(train_df.loc[i, "sentence"]))) for i in longest_rows
)
assert longest_prompt + 2 <= CONFIG["max_length"], (
    f"longest prompt is {longest_prompt} tokens, raise max_length"
)
print(f"longest tokenized prompt: {longest_prompt} tokens, max_length {CONFIG['max_length']} ok")

## Load the 4-bit base model and define the adapter

NF4 quantization with double quantization per the QLoRA paper. Compute
happens in bf16 where the GPU supports it, fp16 on a T4.

In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_id"],
    quantization_config=bnb_config,
    dtype=COMPUTE_DTYPE,
    device_map={"": 0},
    token=HF_TOKEN,
)
model.config.pad_token_id = tok.pad_token_id
model.config.use_cache = False  # required with gradient checkpointing

peft_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)
print(f"4-bit base loaded: {model.get_memory_footprint() / 1e9:.2f} GB on GPU")

## Trainer

`completion_only_loss=True` is explicit even though it is the default for
prompt-completion data. Watch `mean_token_accuracy` in W&B: with one
supervised token per example it is literally the training label accuracy.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=PER_DEVICE,
    per_device_eval_batch_size=PER_DEVICE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type="cosine",
    warmup_ratio=CONFIG["warmup_ratio"],
    bf16=BF16_OK,
    fp16=not BF16_OK,
    max_length=CONFIG["max_length"],
    completion_only_loss=True,
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=CONFIG["epochs"],
    report_to="wandb",
    run_name=f"qlora-r{CONFIG['lora_r']}-{GPU_NAME.replace(' ', '-')}",
    seed=CONFIG["seed"],
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tok,
    peft_config=peft_config,
)

trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer.model.parameters())
print(f"trainable: {trainable:,} of {total:,} params ({100 * trainable / total:.2f}%)")

## Safety cell 2: prove the loss masking before spending GPU time

Pulls one real collated batch and asserts that only the level digit (plus
at most an end-of-text token) carries loss.

In [ ]:
batch = next(iter(trainer.get_train_dataloader()))
supervised = (batch["labels"] != -100).sum(dim=1)
assert 1 <= int(supervised.min()) and int(supervised.max()) <= 2, (
    f"expected 1 or 2 supervised tokens per row, got {supervised.tolist()}"
)
first = batch["labels"][0]
print("supervised tokens per row:", supervised.tolist()[:8])
print("supervised text of first row:", repr(tok.decode(first[first != -100])))
print("Loss lands only on the level digit. Safe to train.")

## Train

About 1,345 optimizer steps for 3 epochs. Reruns of this cell resume from
the last saved checkpoint automatically.

In [ ]:
import glob

has_checkpoint = bool(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
trainer.train(resume_from_checkpoint=True if has_checkpoint else None)

## Pick the best epoch by validation macro F1

Inference is one forward pass per batch: read the logits of the six label
tokens at the final position and take the argmax. Deterministic, no
generation, no parsing. Each epoch checkpoint is scored on the full val
split and the best macro F1 wins.

In [ ]:
import glob

import torch
from sklearn.metrics import accuracy_score, f1_score

from utils import format_prompt

LABEL_ID_LIST = [LABEL_IDS[i] for i in range(1, 7)]


def predict_levels(peft_model, sentences, batch_size=32):
    peft_model.eval()
    tok.padding_side = "left"
    preds = []
    with torch.no_grad():
        for start in range(0, len(sentences), batch_size):
            prompts = [format_prompt(s) for s in sentences[start : start + batch_size]]
            enc = tok(prompts, return_tensors="pt", padding=True).to(peft_model.device)
            logits = peft_model(**enc).logits[:, -1, :]
            preds.extend((logits[:, LABEL_ID_LIST].argmax(dim=-1) + 1).tolist())
    return preds


pm = trainer.model
pm.config.use_cache = True

scores = {}
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[-1]))
for ckpt in checkpoints:
    name = ckpt.rsplit("/", 1)[-1]
    pm.load_adapter(ckpt, adapter_name=name)
    pm.set_adapter(name)
    val_preds = predict_levels(pm, val_df["sentence"].tolist())
    acc = accuracy_score(val_df["label"], val_preds)
    macro = f1_score(val_df["label"], val_preds, average="macro")
    scores[name] = macro
    print(f"{name}: val accuracy {acc:.4f}, val macro F1 {macro:.4f}")

BEST = max(scores, key=scores.get)
BEST_DIR = f"{OUTPUT_DIR}/{BEST}"
pm.set_adapter(BEST)
print("selected checkpoint:", BEST)

## Predict val and test, measure speed, export the shared format

Same schema as the baseline files, including the SHA-256 of the split CSV
so the eval harness can prove both models saw identical sentences.
Throughput is measured batched (the serving cost basis) and latency
single-stream on 100 sentences (the user-experience number).
`parse_failures` is 0 by construction and kept for schema parity.

In [ ]:
import time
from pathlib import Path

import numpy as np

from utils import LABEL_TO_CEFR, save_json, sha256_file


def run_split(split_df, split_name):
    csv_path = Path("data/processed") / f"{split_name}.csv"
    sentences = split_df["sentence"].tolist()

    start = time.perf_counter()
    preds = predict_levels(pm, sentences, batch_size=32)
    wall = time.perf_counter() - start
    throughput = len(sentences) / wall

    sample = split_df.sample(n=min(100, len(split_df)), random_state=CONFIG["seed"])
    latencies = []
    for sentence in sample["sentence"]:
        t0 = time.perf_counter()
        predict_levels(pm, [sentence], batch_size=1)
        latencies.append(time.perf_counter() - t0)

    golds = split_df["label"].tolist()
    within = float(np.mean([abs(g - p) <= 1 for g, p in zip(golds, preds)]))
    payload = {
        "model": "llama-3-8b-qlora",
        "adapter_repo": CONFIG["adapter_repo"],
        "selected_checkpoint": BEST,
        "gpu": GPU_NAME,
        "split": split_name,
        "config": {
            k: CONFIG[k]
            for k in (
                "model_id", "max_length", "lora_r", "lora_alpha", "lora_dropout",
                "epochs", "learning_rate", "effective_batch", "seed",
            )
        },
        "dataset": {"split_csv_sha256": sha256_file(csv_path), "n": len(split_df)},
        "aggregate": {
            "accuracy_pct": round(100 * accuracy_score(golds, preds), 2),
            "within_one_level_pct": round(100 * within, 2),
            "macro_f1": round(float(f1_score(golds, preds, average="macro")), 4),
            "parse_failures": 0,
            "batch_size": 32,
            "batched_throughput_sentences_per_s": round(throughput, 2),
            "latency_s": {
                "mean": round(float(np.mean(latencies)), 3),
                "median": round(float(np.median(latencies)), 3),
                "p95": round(float(np.percentile(latencies, 95)), 3),
            },
        },
        "predictions": [
            {"index": int(i), "gold": LABEL_TO_CEFR[g], "pred": LABEL_TO_CEFR[p]}
            for i, g, p in zip(split_df.index, golds, preds)
        ],
    }
    out = Path("results/predictions") / f"llama3_qlora_{split_name}.json"
    save_json(out, payload)
    agg = payload["aggregate"]
    print(f"{split_name}: accuracy {agg['accuracy_pct']}%, macro F1 {agg['macro_f1']}, wrote {out}")
    return payload


val_payload = run_split(val_df, "val")
test_payload = run_split(test_df, "test")

## Push the selected adapter to the Hugging Face Hub

The repo name starts with Llama-3 and the card says Built with Meta Llama 3,
both required by the Meta license for derivatives.

In [ ]:
from huggingface_hub import HfApi, create_repo

card_lines = [
    "---",
    "base_model: meta-llama/Meta-Llama-3-8B",
    "library_name: peft",
    "license: llama3",
    "language:",
    "- en",
    "tags:",
    "- qlora",
    "- lora",
    "- cefr",
    "- text-classification",
    "---",
    "",
    "# Llama-3-8B-cefr-qlora",
    "",
    "Built with Meta Llama 3.",
    "",
    "QLoRA adapter that rates the CEFR difficulty (A1 to C2) of a single",
    "English sentence. Prompt format, training code, and the full benchmark",
    "against GPT-4o-mini: https://github.com/akallam04/cefr-qlora-benchmark",
    "",
    f"Selected checkpoint {BEST} by validation macro F1 ({scores[BEST]:.4f}).",
    "Test set: accuracy "
    + str(test_payload["aggregate"]["accuracy_pct"])
    + "%, macro F1 "
    + str(test_payload["aggregate"]["macro_f1"]) + ".",
    "",
    "Trained on the public portion of CEFR-SP (Wiki-Auto CC BY-SA 3.0,",
    "SCoRE CC BY-NC-SA 4.0): non-commercial use only for the SCoRE-derived",
    "part.",
]
with open("/content/adapter_readme.md", "w") as fh:
    fh.write("\n".join(card_lines) + "\n")

api = HfApi(token=HF_TOKEN)
create_repo(CONFIG["adapter_repo"], exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=BEST_DIR,
    repo_id=CONFIG["adapter_repo"],
    allow_patterns=["adapter_model.safetensors", "adapter_config.json"],
)
api.upload_file(
    path_or_fileobj="/content/adapter_readme.md",
    path_in_repo="README.md",
    repo_id=CONFIG["adapter_repo"],
)
print(f"adapter pushed: https://huggingface.co/{CONFIG['adapter_repo']}")

## Export the prediction files

Download both files and place them in `results/predictions/` in the local
repo, then continue with the eval harness phase.

In [ ]:
import wandb
from google.colab import files

if wandb.run is not None:
    print("wandb run:", wandb.run.url)
    wandb.finish()

files.download("results/predictions/llama3_qlora_val.json")
files.download("results/predictions/llama3_qlora_test.json")
print("Place both files in results/predictions/ of the local repo.")